# Project Writers' Room: Module 3
## Lexical Diversity & Vocabulary Saturation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evecount/writersroom_NLP_workbook/blob/main/notebooks/03_lexical_richness_zipf.ipynb)

### Learning Objectives
1. Differentiate between Types ($V$) and Tokens ($N$).
2. Compute standard Type-Token Ratio (TTR) and sample-size invariant Root TTR ($V / \sqrt{N}$).
3. Mine Hapax Legomena (terms occurring exactly once) to isolate signature character jargon.
4. Validate Zipf's Law on sitcom dialogue through log-log linear regression.


In [ ]:
import sys
sys.path.insert(0, '..')
from writersroom import LexicalAnalyzer, ColloquialTokenizer
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_json('../data/processed/dialogue_master.json')
tok = ColloquialTokenizer()
print(f'Loaded {len(df)} total dialogue lines.')


### 1. Character Vocabulary Breadth Comparison
We isolate tokens spoken by **Sheldon Cooper** vs. **Joey Tribbiani** vs. **Chandler Bing**.


In [ ]:
def get_character_tokens(speaker_name):
    lines = df[df['speaker'] == speaker_name]['line']
    text = ' '.join(lines)
    return tok.tokenize_colloquial(text)

sheldon_tokens = get_character_tokens('SHELDON')
joey_tokens = get_character_tokens('JOEY')
chandler_tokens = get_character_tokens('CHANDLER')

print(f'Sheldon  Tokens: {len(sheldon_tokens)}, Types: {len(set(sheldon_tokens))}, TTR: {LexicalAnalyzer.compute_ttr(sheldon_tokens)}')
print(f'Joey     Tokens: {len(joey_tokens)}, Types: {len(set(joey_tokens))}, TTR: {LexicalAnalyzer.compute_ttr(joey_tokens)}')
print(f'Chandler Tokens: {len(chandler_tokens)}, Types: {len(set(chandler_tokens))}, TTR: {LexicalAnalyzer.compute_ttr(chandler_tokens)}')


### 2. Hapax Legomena: Academic Jargon vs. Everyday Speech
Hapaxes ($f=1$) reveal idiosyncratic character lexicon.


In [ ]:
print('Sheldon Hapaxes (sample):', LexicalAnalyzer.find_hapaxes(sheldon_tokens)[:10])
print('Joey Hapaxes (sample):', LexicalAnalyzer.find_hapaxes(joey_tokens)[:10])


### 3. Zipf's Law: Empirical Log-Log Fit
Zipf's law states that word frequency is inversely proportional to rank:
$$f(r) \propto \frac{1}{r^s} \iff \log f = \log C - s \log r$$


In [ ]:
all_tokens = tok.tokenize_colloquial(' '.join(df['line']))
zipf_data = LexicalAnalyzer.fit_zipf_law(all_tokens)
print(f'Fitted Slope: {zipf_data["slope"]}, Intercept: {zipf_data["intercept"]}, R^2: {zipf_data["r_squared"]}')

# Log-Log Plot
import numpy as np
plt.figure(figsize=(8, 5))
plt.scatter(np.log(zipf_data['ranks'][:50]), np.log(zipf_data['freqs'][:50]), color='crimson', label='Empirical Tokens')
plt.title("Zipf's Law in Sitcom Corpus (Log-Log Scale)")
plt.xlabel('Log(Rank)')
plt.ylabel('Log(Frequency)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


### 4. Self-Verification & Autograding


In [ ]:
from tests.autograde_checks import check_module_3
check_module_3()
